# Preprocessing Pipeline
**Notebook 02 — IDX Exchange Data Science Internship 2026**

This notebook prepares the CRMLS dataset for machine learning modeling. The raw data 
contains 80+ fields per listing across 400,000+ records. This pipeline filters, cleans, 
encodes, normalizes, and splits the data into train and test sets ready for modeling.

**Input:** Raw CRMLS monthly CSV files (January 2022 – June 2026)  
**Output:** `train_cleaned.csv` and `test_cleaned.csv`

In [18]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

filepath = r"C:\Users\julia\downloads\IDX_summer_internship"
pattern = os.path.join(filepath, "CRMLSSold*.csv")
files = sorted(glob.glob(pattern))

# load and concatenate all files at once
df_list = [pd.read_csv(f, low_memory=False) for f in files]
data = pd.concat(df_list, ignore_index=True)

print(f"Loaded {len(files)} files")
print(f"Total rows: {len(data)}")
print(f"Columns: {data.shape[1]}")

Loaded 31 files
Total rows: 818778
Columns: 82


In [20]:
# print starting columns
filtered_data = data[(data['PropertyType'] == 'Residential') & (data['PropertySubType'] == 'SingleFamilyResidence')].copy()
filtered_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 412015 entries, 3 to 818775
Data columns (total 82 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   Flooring                      264736 non-null  object 
 1   ViewYN                        371253 non-null  object 
 2   WaterfrontYN                  186 non-null     object 
 3   BasementYN                    10069 non-null   object 
 4   PoolPrivateYN                 370408 non-null  object 
 5   OriginalListPrice             411209 non-null  float64
 6   ListingKey                    412015 non-null  int64  
 7   ListAgentEmail                411137 non-null  object 
 8   CloseDate                     412015 non-null  object 
 9   ClosePrice                    412013 non-null  float64
 10  ListAgentFirstName            408515 non-null  object 
 11  ListAgentLastName             411978 non-null  object 
 12  Latitude                      411822 non-null  fl

## 1. Missing Value Handling

Columns where more than 50% of data is missing are removed entirely as imputing beyond 
this threshold would introduce more synthetic values than real signal. For remaining 
columns, numeric fields are imputed with the median (robust to outliers) and 
categorical fields with the mode (most frequent value). Columns with only one 
unique value are also dropped as they carry no information for modeling.

In [25]:
# remove columns with 100 % missing data
data_clean = filtered_data.dropna(axis=1, how="all")
print(f"Columns remaining after 100% drop: {data_clean.shape[1]}")

Columns remaining after 100% drop: 74


In [27]:
# removing columns with more than 50% missing
threshold = 0.5
data_clean = data_clean.loc[:, data_clean.isnull().mean() < threshold]
print(f"Columns remaining after 50% drop: {data_clean.shape[1]}")

Columns remaining after 50% drop: 55


In [29]:
# imputing remaining missing values
for col in data_clean.columns:
    if data_clean[col].dtype in ['float64', 'int64']:
        # where numeric -> impute with median
        data_clean[col] = data_clean[col].fillna(data_clean[col].median())
    else:
        # where categorical impute with mode
        data_clean[col] = data_clean[col].fillna(data_clean[col].mode()[0])

C:\Users\julia\AppData\Local\Temp\ipykernel_8684\1140446256.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_clean[col] = data_clean[col].fillna(data_clean[col].mode()[0])


## 2. Data Type Conversion

Each column is converted to its appropriate type before modeling:
- **Date columns** converted to datetime format to support time-based splitting
- **Integer-like features** (bedrooms, bathrooms, year built, stories, garage spaces) converted to Int64
- **PostalCode** converted to string, as zip codes are geographic labels not numeric quantities
- **GarageSpaces / ParkingTotal** rounded to integer (half-values like 2.5 are not meaningful)
- **Boolean columns** converted to binary integers (0/1)

In [37]:
# converting date columns to datetime
date_cols = ['CloseDate', 'ListingContractDate', 
             'ContractStatusChangeDate', 'PurchaseContractDate']
for col in date_cols:
    if col in data_clean.columns:
        data_clean[col] = pd.to_datetime(data_clean[col], errors='coerce')

In [41]:
# converting integer-like features to integer format
int_cols = ['BedroomsTotal', 'BathroomsTotalInteger', 'YearBuilt', 'Stories', 'MainLevelBedrooms', 'GarageSpaces', 'ParkingTotal']

for col in int_cols:
    if col in data_clean.columns:
        data_clean[col] = pd.to_numeric(data_clean[col], errors='coerce').round().astype('Int64')

print(data_clean[int_cols].dtypes)

BedroomsTotal            Int64
BathroomsTotalInteger    Int64
YearBuilt                Int64
Stories                  Int64
MainLevelBedrooms        Int64
GarageSpaces             Int64
ParkingTotal             Int64
dtype: object


In [43]:
# dropping columns that are redundant with lat/long
data_clean = data_clean.drop(columns=['Flooring', 'City', 'CountyOrParish', 'MLSAreaMajor', 'PostalCode', 'ListAgentAOR', 
                                      'ContractStatusChangeDate', 'ListingContractDate', 'BuyerOfficeAOR', 'BuyerAgentAOR', 'ListingKey'])

## 3. Removing Redundant Variables & Invalid Observations

Columns that are redundant with latitude/longitude (City, CountyOrParish) and 
high-cardinality identifier columns (listing agent names, addresses) are dropped.

Rows with logically impossible values are removed:
- Invalid or zero ClosePrice
- Unrealistic living area (< 200 or > 15,000 sq ft)
- Invalid bedroom/bathroom counts (0 or > 15)
- Logical impossibility: 5+ bedrooms in under 800 sq ft
- Invalid year built (outside 1800–2026)
- Duplicate transactions (same CloseDate, ClosePrice, LivingArea, location)
- Invalid coordinates (latitude/longitude outside valid ranges or equal to zero)

In [46]:
# Remove invalid observations
initial_count = len(data_clean)

# invalid ClosePrice
data_clean = data_clean[data_clean['ClosePrice'] > 0]
data_clean = data_clean.dropna(subset=['ClosePrice'])

# unrealistic ClosePrice range
data_clean = data_clean[
    (data_clean['ClosePrice'] >= 50000) & 
    (data_clean['ClosePrice'] <= 10000000)
]

# invalid living area
data_clean = data_clean[
    (data_clean['LivingArea'] >= 200) & 
    (data_clean['LivingArea'] <= 15000)
]

# invalid bedroom/bathroom counts
data_clean = data_clean[
    (data_clean['BedroomsTotal'] > 0) & 
    (data_clean['BedroomsTotal'] <= 15) &
    (data_clean['BathroomsTotalInteger'] > 0) & 
    (data_clean['BathroomsTotalInteger'] <= 15)
]

# logical impossibility: too many bedrooms for living area
data_clean = data_clean[
    ~((data_clean['BedroomsTotal'] >= 5) & 
      (data_clean['LivingArea'] < 800))
]

# invalid year built
data_clean = data_clean[
    (data_clean['YearBuilt'] >= 1800) & 
    (data_clean['YearBuilt'] <= 2026)
]

# duplicate transactions - using specific combination to avoid removing legitimate sales
data_clean = data_clean.drop_duplicates(
    subset=['CloseDate', 'ClosePrice', 'LivingArea', 'BedroomsTotal', 'Latitude', 'Longitude']
)

# summary
removed = initial_count - len(data_clean)
print(f"Rows removed: {removed:,} ({removed / initial_count * 100:.2f}%)")
print(f"Remaining rows: {len(data_clean):,}")

Rows removed: 2,093 (0.51%)
Remaining rows: 409,922


In [48]:
# Converting Boolean to binary
bool_cols = ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN']

for col in bool_cols:
    if col in data_clean.columns:
        data_clean[col] = data_clean[col].astype(int)

print(data_clean[bool_cols].dtypes)

ViewYN               int32
PoolPrivateYN        int32
AttachedGarageYN     int32
FireplaceYN          int32
NewConstructionYN    int32
dtype: object


In [49]:
# drop columns with only 1 unique value, dropping columns with non-useful object columns
single_value_cols = [col for col in data_clean.columns 
                     if data_clean[col].nunique() == 1]
print(f"Dropping single value columns: {single_value_cols}")
data_clean = data_clean.drop(columns=single_value_cols)

# drop high cardinality and non-useful object columns
drop_cols = [
    'ListAgentEmail', 'ListAgentFirstName', 'ListAgentLastName',
    'ListAgentFullName', 'ListOfficeName', 'BuyerOfficeName',
    'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName',
    'UnparsedAddress', 'ListingId', 'Flooring', 'MLSAreaMajor', 
    'ContractStatusChangeDate', 'PurchaseContractDate', 
    'ListingContractDate'
]
drop_cols = [c for c in drop_cols if c in data_clean.columns]
data_clean = data_clean.drop(columns=drop_cols)
print(f"Shape after dropping: {data_clean.shape}")

Dropping single value columns: ['PropertyType', 'MlsStatus', 'PropertySubType']
Shape after dropping: (409922, 29)


In [52]:
# checking remaining columns before encoding
obj_cols = data_clean.select_dtypes(include='object').columns
for col in obj_cols:
    print(f"{col}: {data_clean[col].nunique()} unique values")\

data_clean.drop(columns='HighSchoolDistrict')
print(data_clean.columns)

StateOrProvince: 13 unique values
Levels: 17 unique values
HighSchoolDistrict: 447 unique values
Index(['ViewYN', 'PoolPrivateYN', 'OriginalListPrice', 'CloseDate',
       'ClosePrice', 'Latitude', 'Longitude', 'LivingArea', 'ListPrice',
       'DaysOnMarket', 'ListingKeyNumeric', 'AttachedGarageYN', 'ParkingTotal',
       'LotSizeAcres', 'YearBuilt', 'StreetNumberNumeric',
       'BathroomsTotalInteger', 'BedroomsTotal', 'StateOrProvince',
       'FireplaceYN', 'Stories', 'Levels', 'LotSizeArea', 'MainLevelBedrooms',
       'NewConstructionYN', 'GarageSpaces', 'HighSchoolDistrict',
       'AssociationFee', 'LotSizeSquareFeet'],
      dtype='object')


In [58]:
# converting impossible coordinates to NaNs
invalid_coords = (
    (data_clean['Latitude'] < -90) | (data_clean['Latitude'] > 90) |
    (data_clean['Longitude'] < -180) | (data_clean['Longitude'] > 180) |
    (data_clean['Latitude'] == 0) | (data_clean['Longitude'] == 0)
)
data_clean.loc[invalid_coords, ['Latitude', 'Longitude']] = np.nan

# then drop rows with missing coordinates
data_clean = data_clean.dropna(subset=['Latitude', 'Longitude'])

## 4. Encoding

Machine learning models require numeric input, categorical text columns must be 
converted to numbers. One-hot encoding is applied to categorical columns with 
fewer than 100 unique values, creating a binary indicator column for each category. 
Columns with 100+ unique values are dropped as encoding them would create too many 
sparse columns with little predictive value.

Note: categories are standardized (stripped and uppercased) before encoding to 
ensure consistency.

In [60]:
# one hot encoding

# Select categorical columns to encode
ohe_cols = [
    col for col in data_clean.select_dtypes(include="object").columns
    if data_clean[col].nunique() < 100
]
# Standardize categories
for col in ohe_cols:
    data_clean[col] = (
        data_clean[col]
        .astype(str)
        .str.strip()
        .str.upper()
    )

# One-hot encode
data_clean = pd.get_dummies(
    data_clean,
    columns=ohe_cols,
    drop_first=True
)

print(data_clean.shape)

(409880, 55)


## 5. Train/Test Split

Rather than splitting randomly, a time-based chronological split is used to 
mirror real-world prediction conditions so that the model always predicts future sales 
based on historical data.

- **Test set:** Most recent month of available data (June 2026)
- **Training set:** X preceding months (X is a tunable parameter)
- **Training window used:** 30 months

This approach prevents future data from leaking into the training set and ensures 
evaluation reflects true out-of-sample performance.

In [66]:
def make_train_test_split(df, date_col='CloseDate', x_months=12):
    """
    Creates a time-based train/test split.
    
    Parameters:
        df: cleaned dataframe
        date_col: name of the date column to split on
        x_months: number of months to use for training window (tunable)
    
    Returns:
        train_df, test_df
    """
    df[date_col] = pd.to_datetime(df[date_col], format='mixed')
    
    latest_month = df[date_col].dt.to_period('M').max()
    cutoff_month = latest_month - x_months
    
    test_df = df[df[date_col].dt.to_period('M') == latest_month].copy()
    train_df = df[
        (df[date_col].dt.to_period('M') > cutoff_month) &
        (df[date_col].dt.to_period('M') < latest_month)
    ].copy()
    
    print(f"Test month: {latest_month}")
    print(f"Training window: {x_months} months ({cutoff_month} to {latest_month})")
    print(f"Training rows: {len(train_df):,}")
    print(f"Testing rows:  {len(test_df):,}")
    
    return train_df, test_df

train_df, test_df = make_train_test_split(data_clean, x_months=30)


Test month: 2026-06
Training window: 30 months (2023-12 to 2026-06)
Training rows: 318,860
Testing rows:  12,777


## 6. Outlier Removal Using Training Thresholds

To remove data entry errors and extreme outliers without introducing data leakage, 
cutoff thresholds are computed from the training set only and applied as frozen 
values to both train and test sets. The test set's own distribution is never used 
to compute thresholds.

Two filters are applied:
1. **ClosePrice filter** where properties below the 0.5th percentile or above the 99.5th 
   percentile of training ClosePrice are removed
2. **Price-per-square-foot filter** where a secondary filter catches data entry errors 
   that a raw price filter misses (e.g. a reasonably priced home with an incorrectly 
   entered square footage)

In [72]:
# outlier removal using training thresholds

# Remove extreme ClosePrice outliers
# thresholds learned from training data only

# compute thresholds from training set only
lower_limit = train_df["ClosePrice"].quantile(0.005)
upper_limit = train_df["ClosePrice"].quantile(0.995)

print(f"Lower limit (0.5th percentile): ${lower_limit:,.0f}")
print(f"Upper limit (99.5th percentile): ${upper_limit:,.0f}")

# apply same thresholds to both sets
before_train = len(train_df)
before_test = len(test_df)

train_df = train_df[
    (train_df["ClosePrice"] >= lower_limit) &
    (train_df["ClosePrice"] <= upper_limit)
]

test_df = test_df[
    (test_df["ClosePrice"] >= lower_limit) &
    (test_df["ClosePrice"] <= upper_limit)
]

print(f"Train removed: {before_train - len(train_df):,} rows")
print(f"Test removed: {before_test - len(test_df):,} rows")
print(f"Train remaining: {len(train_df):,}")
print(f"Test remaining: {len(test_df):,}")


Lower limit (0.5th percentile): $195,300
Upper limit (99.5th percentile): $6,700,000
Train removed: 3,177 rows
Test removed: 143 rows
Train remaining: 315,683
Test remaining: 12,634


## 7. Normalization

Continuous features are standardized using StandardScaler, transforming each 
variable to have a mean of zero and standard deviation of one. This ensures features 
on different scales contribute equally to the model rather than allowing 
large-magnitude features to dominate. For example, when using Linear Regression, the
fit would be biased if the data were not noramlized.

**Important:** `fit_transform` is applied to the training set only. The scaler 
learns the mean and standard deviation from training data and applies those same 
frozen values to the test set, this prevents any leakage of test distribution 
information into the scaling process.

Original latitude and longitude values are saved before scaling as `Latitude_orig` 
and `Longitude_orig` for use in the geographic feature engineering step in Notebook 04.

In [77]:
from sklearn.preprocessing import StandardScaler

continuous_columns = [
    "Latitude", "Longitude", "LivingArea", "DaysOnMarket",
    "ParkingTotal", "LotSizeAcres", "YearBuilt",
    "StreetNumberNumeric", "BathroomsTotalInteger",
    "BedroomsTotal", "Stories", "LotSizeArea",
    "MainLevelBedrooms", "GarageSpaces",
    "AssociationFee", "LotSizeSquareFeet"
]

# Only use columns that still exist
continuous_columns = [
    col for col in continuous_columns
    if col in train_df.columns
]

# save original coordinates before scaling
train_df["Latitude_orig"] = train_df["Latitude"]
train_df["Longitude_orig"] = train_df["Longitude"]
test_df["Latitude_orig"] = test_df["Latitude"]
test_df["Longitude_orig"] = test_df["Longitude"]

scaler = StandardScaler()

train_df[continuous_columns] = scaler.fit_transform(
    train_df[continuous_columns]
)

test_df[continuous_columns] = scaler.transform(
    test_df[continuous_columns]
)

print(f"Scaled {len(continuous_columns)} continuous features.")

Scaled 16 continuous features.


In [51]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print(
    "Remaining object columns:",
    train_df.select_dtypes(include="object").columns.tolist()
)

print(
    "Postal code columns present:",
    any(col.startswith("PostalCode") for col in train_df.columns)
)

Train shape: (315683, 57)
Test shape: (12634, 57)
Remaining object columns: ['HighSchoolDistrict']
Postal code columns present: False


## 8. Export

Cleaned train and test datasets are exported as CSV files for use in downstream 
modeling notebooks.

> **Note:** These files are not tracked in the GitHub repository due to file size and for data privacy. 
> Rerun this notebook to regenerate them locally.

In [83]:
# export cleaned train/test splits
train_df.to_csv(filepath + r"\train_cleaned.csv", index=False)
test_df.to_csv(filepath + r"\test_cleaned.csv", index=False)

print(f"train_cleaned.csv saved — {len(train_df):,} rows")
print(f"test_cleaned.csv saved — {len(test_df):,} rows")

train_cleaned.csv saved — 315,683 rows
test_cleaned.csv saved — 12,634 rows


## Final Dataset Summary

**Training observations:** 303,839  
**Testing observations:** 12,634  
**Features after preprocessing:** 40  

### Preprocessing Steps Applied
1. Filtered to single-family residential properties (PropertyType = Residential, PropertySubType = SingleFamilyResidence)
2. Removed columns with 100% missing values
3. Removed columns with more than 50% missing data
4. Imputed remaining missing values (median for numeric, mode for categorical)
5. Converted data types (datetime, integer, string, boolean to binary)
6. Removed redundant variables and invalid observations (invalid ClosePrice, unrealistic living area, impossible bedroom counts, duplicate transactions, invalid coordinates)
7. One-hot encoding of categorical columns with fewer than 100 unique values
8. Outlier removal using training data thresholds (0.5th–99.5th percentile ClosePrice and price-per-sqft)
9. Feature scaling using StandardScaler (fit on training data only)
10. Time-based train/test split (June 2026 reserved as test set)

> Note: Feature count increases to 52 after feature engineering in Notebook 04